In [5]:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime
import requests_cache
from tqdm import tqdm


In [3]:
cache_name = 'yfinance_cache'
expire_after = datetime.timedelta(days=1) # Cache expires after 1 day

# Create a cached session
session = requests_cache.CachedSession(
    cache_name=cache_name,
    backend='sqlite',
    expire_after=expire_after
)

In [15]:
class Dataloader:
    def __init__(self, period, session, extra_obs, max_episode_length=1000):
        self.session = session
        self.period = period
        self.extra_obs = extra_obs
        self.max_episode_length = max_episode_length
        self.vix_data = yf.Ticker("^VIX", session=self.session).history(period=period, auto_adjust=True)
        self.gspc_data = yf.Ticker("^GSPC", session=self.session).history(period=period, auto_adjust=True)
        self.random_symbols = [
            "NVDA", "AMZN", "GOOGL", "MSFT", "AAPL", "META", "ADBE",
            "NFLX", "TSLA", "JPM", "V", "UNH"
        ]
        
        self.random_symbols_extend = [
            # Communication Services
            "T", "VZ", "META", "CMCSA",
            # Consumer Discretionary
            "AMZN", "TSLA", "NKE", "MCD", "HD",
            # Consumer Staples
            "PG", "KO", "PEP", "WMT",
            # Energy
            "XOM", "CVX", "COP", "SLB", "OXY",
            # Financials
            "JPM", "BAC", "WFC", "GS", "C",
            # Healthcare
            "JNJ", "PFE", "MRK", "UNH", "ABT",
            # Industrials
            "BA", "CAT", "HON", "MMM", "UNP",
            # Information Technology
            "AAPL", "MSFT", "GOOGL", "NVDA", "ADBE",
            # Materials
            "SHW", "DD", "LIN", "NEM",
            # Real Estate
            "AMT", "PLD", "SPG", "AVB",
            # Utilities
            "NEE", "DUK", "SO", "D"
        ]
    
    def dataloader(self): 
        all_stock_data = {}
        for symbol in tqdm(self.random_symbols_extend, desc="Fetching stock data"):
                all_stock_data[symbol] = self.batch_fetch_data(symbol)
        return all_stock_data
    
    def batch_fetch_data(self, stock):
        # print(f"[INFO] Fetching data for {stock}...")
        try:
            stock_ticker = yf.Ticker(stock, session=self.session)
            stock_data = stock_ticker.history(period=self.period, auto_adjust=True)
            if self.extra_obs:
                stock_df = stock_data[['Open', 'High', 'Low', 'Close', 'Volume']].copy() if not stock_data.empty else pd.DataFrame(index=stock_data.index)
                vix_df = self.vix_data[['Close']].rename(columns={'Close': 'VIX_Close'}).copy() if not self.vix_data.empty else pd.DataFrame(index=self.vix_data.index)
                gspc_df = self.gspc_data[['Close']].rename(columns={'Close': 'GSPC_Close'}).copy() if not self.gspc_data.empty else pd.DataFrame(index=self.gspc_data.index)
                merged_data = pd.concat([stock_df, vix_df, gspc_df], axis=1, join='outer')
                if 'Close' not in merged_data.columns and not stock_df.empty:
                    print("[ERROR] Primary stock 'Close' column missing after outer join.")
            else:
                stock_df = stock_data[['Open', 'High', 'Low', 'Close', 'Volume']].copy() if not stock_data.empty else pd.DataFrame(index=stock_data.index)
                merged_data = stock_df
            merged_data = merged_data.groupby(merged_data.index.date).first()
            merged_data.index = pd.to_datetime(merged_data.index) # Ensure index is datetime
            
            merged_data = merged_data.ffill(limit=3)
            # Handle NaNs in critical columns
            if 'Volume' in merged_data.columns:
                merged_data['Volume'].fillna(0.0, inplace=True) # Fill NaN volume with 0
            merged_data.dropna(subset=['Close'], inplace=True) # Drop rows ONLY if 'Close' is missing
            min_required_length = 35 + self.max_episode_length
            if len(merged_data) < min_required_length:
                print(f"[WARNING] Insufficient merged data after processing for {stock} (Final Length: {len(merged_data)}, Required: {min_required_length}).")
                return None
            
            return merged_data
        except Exception as e:
            print(f"[ERROR] Exception during data fetch/process for {stock}: {e}")
            return None

dataloader = Dataloader(period="max", session=session, extra_obs=True)
dataloader = dataloader.dataloader()

Fetching stock data:   0%|          | 0/50 [00:00<?, ?it/s]/var/folders/qw/5rcwl4d96zvcq7zxv7fzt3m80000gn/T/ipykernel_16136/3866314840.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data['Volume'].fillna(0.0, inplace=True) # Fill NaN volume with 0
Fetching stock data:   2%|▏         | 1/50 [00:00<00:16,  2.91it/s]/var/folders/qw/5rcwl4d96zvcq7zxv7fzt3m80000gn/T/ipykernel_16136/3866314840.py:66: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace

In [17]:
#find total number of rows in the data
total_rows = 0
for stock, data in dataloader.items():
    if data is not None:
        total_rows += len(data)
        print(f"[INFO] {stock} has {len(data)} rows of data.")
    else:
        print(f"[INFO] {stock} has no valid data.")
print(f"[INFO] Total rows of data across all stocks: {total_rows}")

[INFO] T has 10439 rows of data.
[INFO] VZ has 10439 rows of data.
[INFO] META has 3253 rows of data.
[INFO] CMCSA has 11371 rows of data.
[INFO] AMZN has 7031 rows of data.
[INFO] TSLA has 3730 rows of data.
[INFO] NKE has 11191 rows of data.
[INFO] MCD has 14801 rows of data.
[INFO] HD has 10988 rows of data.
[INFO] PG has 15936 rows of data.
[INFO] KO has 15936 rows of data.
[INFO] PEP has 13337 rows of data.
[INFO] WMT has 13277 rows of data.
[INFO] XOM has 15936 rows of data.
[INFO] CVX has 15936 rows of data.
[INFO] COP has 10918 rows of data.
[INFO] SLB has 10918 rows of data.
[INFO] OXY has 10918 rows of data.
[INFO] JPM has 11371 rows of data.
[INFO] BAC has 13157 rows of data.
[INFO] WFC has 13337 rows of data.
[INFO] GS has 6536 rows of data.
[INFO] C has 12180 rows of data.
[INFO] JNJ has 15936 rows of data.
[INFO] PFE has 13337 rows of data.
[INFO] MRK has 15936 rows of data.
[INFO] UNH has 10210 rows of data.
[INFO] ABT has 11371 rows of data.
[INFO] BA has 15936 rows of 